# Satisfaction time: *when* does a constraint first hold?

`viterbi_torch_mvr_chmm` and `marginal_map_torch_mvr_chmm` both decode hidden
states subject to constraints that are **required to hold**. This notebook asks a
different kind of question, about a constraint rather than about the states:

> Given the observations, and given that every *other* constraint holds, when does
> this one first become satisfied?

The answer is a distribution over times. For a target MVR enforced on
$[a, b]$, `sat_time_torch_mvr_chmm` returns

$$w[t] \;=\; P\big(y,\ \text{other constraints hold},\ \text{target's } \texttt{evl}
\text{ FIRST holds at } t\big)$$

normalized over $t \in [a, b]$. Because only the "not yet satisfied" branch is
carried forward, a path is counted at the **earliest** time its target accepts,
and a path whose target never accepts inside the window contributes nothing.

Two things follow that are easy to get wrong, and both are shown below:

1. This is **not** $P(\text{satisfied at } t)$ evaluated time by time.
2. The target is **not** required to hold at the end of its window — unlike every
   constraint in ordinary constrained inference.

In [ ]:
import itertools
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.other_queries.sat_time_mvr import (
    sat_time_torch_mvr_chmm,
)
from conin.hidden_markov_model.learning.baum_welch_mvr import (
    forward_backward_mvr_chmm,
)

## 1. The model

The same three-state HMM and observation sequence as `MVR_viterbi.ipynb` and
`MVR_marginal_map.ipynb`, so the notebooks can be read against each other.

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

start_probs = {
    "A": 0.2765440507007986,
    "B": 0.4033576072467887,
    "C": 0.32009834205241255,
}

transition_probs = {
    ("A", "A"): 0.3391777054270445,
    ("A", "B"): 0.049711711669595204,
    ("A", "C"): 0.6111105829033604,
    ("B", "A"): 0.48102507253852517,
    ("B", "B"): 0.05601918704283972,
    ("B", "C"): 0.4629557404186351,
    ("C", "A"): 0.43616112524444134,
    ("C", "B"): 0.1773076392327265,
    ("C", "C"): 0.38653123552283214,
}

emission_probs = {
    ("A", "lo"): 0.19949219710155375,
    ("A", "mid"): 0.30789837305397333,
    ("A", "hi"): 0.492609429844473,
    ("B", "lo"): 0.534907622618408,
    ("B", "mid"): 0.234417585356662,
    ("B", "hi"): 0.23067479202493,
    ("C", "lo"): 0.09093879934300991,
    ("C", "mid"): 0.008996844382398088,
    ("C", "hi"): 0.9000643562745919,
}

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)

print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## 2. The target constraint

A *reach* MVR: it accepts as soon as the path has visited a chosen state, and
once accepting it stays accepting. `B` is a good target here — the transitions
into it are small (0.05, 0.06, 0.18), so reaching it is genuinely uncertain and
the satisfaction time spreads out over the horizon.

In [ ]:
def reach_mvr(state, time_range=None, name=None):
    """MVR accepting once ``state`` has been visited; acceptance is absorbing."""
    mediation_states = ["not_yet", "seen"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("seen" if h == state else "not_yet") for h in HIDDEN_STATES},
        upd={
            (m, h): ("seen" if m == "seen" or h == state else "not_yet")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"not_yet": False, "seen": True},
        time_range=time_range,
        name=name,
    )


reach_B = reach_mvr("B", name="reach_B")
model = MVR_CHMM(hidden_markov_model=hmm, constraints=[reach_B])

times, probs = sat_time_torch_mvr_chmm(model, observed, target="reach_B")

print("times :", times)
print("probs :", np.round(probs.numpy(), 4), " sum =", round(float(probs.sum()), 12))

## 3. Checking it against brute force

$T = 7$, so all $3^7 = 2187$ hidden paths can be enumerated. The definition is
direct: weight each path, find the first time it visits `B`, and add its weight
into that bucket. Paths that never visit `B` are dropped.

In [ ]:
def path_prob(path):
    """Joint probability of a hidden path and the observations."""
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = math.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += math.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in enumerate(observed):
        total += math.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return math.exp(total)


first_hit = np.zeros(T)   # P(y, first visit to B at t)
ever_by_t = np.zeros(T)   # P(y, B visited at or before t)
evidence = 0.0

for path in itertools.product(HIDDEN_STATES, repeat=T):
    w = path_prob(path)
    evidence += w
    seen = False
    for t in range(T):
        if path[t] == "B" and not seen:
            first_hit[t] += w
            seen = True
        if seen:
            ever_by_t[t] += w

brute = first_hit / first_hit.sum()

print("brute force :", np.round(brute, 4))
print("sat_time    :", np.round(probs.numpy(), 4))
print("max abs diff:", f"{np.abs(brute - probs.numpy()).max():.3e}")

## 4. The headline: first satisfaction is not per-time satisfaction

These are two different curves and they answer two different questions.

- **First satisfaction** is a *stopping time*. It sums to 1 over the window and
  is what `sat_time_torch_mvr_chmm` returns.
- **$P(\text{satisfied at } t)$** is monotone increasing here, because this
  target's acceptance is absorbing. It does not sum to anything in particular.

For an absorbing target the second curve is the running total of the first,
scaled by the probability that the target is ever satisfied at all. For a target
that can switch back out of accepting — a parity constraint, say — even that
relationship breaks down.

In [ ]:
sat_at_t = ever_by_t / evidence

fig, ax = plt.subplots(figsize=(9, 4.2))

ax.bar(times, probs.numpy(), color="#2E6DB4", label="first satisfaction (sums to 1)")
ax.plot(
    times, sat_at_t, "o-", color="#E08A3C", linewidth=2,
    label=r"$P(\mathrm{satisfied\ at\ }t)$",
)
ax.axhline(
    ever_by_t[-1] / evidence, color="0.55", linestyle="--", linewidth=1,
    label="P(B ever visited)",
)

ax.set_xlabel("t")
ax.set_ylabel("probability")
ax.set_title(
    "Two different questions about the same constraint\n"
    "blue = when does it FIRST hold    orange = does it hold at t",
    loc="left", fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

## 5. The unnormalized weights carry the evidence

`return_log_weights=True` also gives the weights before normalization. Their
`logsumexp` is

$$\log P(y,\ \text{other constraints hold},\ \text{target satisfied somewhere in } [a,b])$$

For an absorbing target, "satisfied somewhere in $[a, b]$" and "satisfied at $b$"
are the same event — so this total must equal the constrained log-likelihood that
`forward_backward_mvr_chmm` reports when the same MVR is enforced as an ordinary
constraint. That is a check against a completely separate implementation.

In [ ]:
_, _, log_weights = sat_time_torch_mvr_chmm(
    model, observed, target="reach_B", return_log_weights=True
)

total = float(torch.logsumexp(log_weights, dim=0))
_, _, loglik = forward_backward_mvr_chmm(model, observed)

print(f"logsumexp(log_weights)        = {total:.12f}")
print(f"forward_backward loglik       = {loglik:.12f}")
print(f"difference                    = {abs(total - loglik):.2e}")
print()
print(f"P(B ever visited | y)         = {math.exp(total) / evidence:.6f}")
print(f"brute force                   = {ever_by_t[-1] / evidence:.6f}")

## 6. Conditioning on another constraint

The target is one constraint among several; the rest are enforced normally and
change the answer. Here `forbid_A` rules out any path visiting `A`, which
removes a whole family of routes to `B` and pushes the satisfaction time earlier.

Note the two ways of naming the target: by list index, or by the `name` given to
the MVR at construction. The second is what keeps this readable once there are
several constraints.

In [ ]:
def forbid_mvr(state, time_range=None, name=None):
    """MVR rejecting any path that visits ``state``."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
        name=name,
    )


both = MVR_CHMM(
    hidden_markov_model=hmm,
    constraints=[forbid_mvr("A", name="forbid_A"), reach_B],
)

_, conditioned = sat_time_torch_mvr_chmm(both, observed, target="reach_B")
_, by_index = sat_time_torch_mvr_chmm(both, observed, target=1)
_, by_negative = sat_time_torch_mvr_chmm(both, observed, target=-1)

assert np.allclose(conditioned.numpy(), by_index.numpy())
assert np.allclose(conditioned.numpy(), by_negative.numpy())

print(pd.DataFrame({
    "t": times,
    "reach_B alone": np.round(probs.numpy(), 4),
    "and forbid_A": np.round(conditioned.numpy(), 4),
}).to_string(index=False))

## 7. Windowing initializes, it does not truncate

A `time_range` on the target narrows the support, but **not** by slicing the
full-horizon answer. Per the `time_range` semantics used everywhere in this
package, the automaton is *initialized at the window start*: with
`time_range=[3, 6]` a fresh `reach_B` begins at `t = 3` and never sees
`hidden[0..2]`, so a path that visited `B` early and not again counts as never
satisfied.

The two columns below therefore differ. If what you want is the full-horizon
distribution restricted to late times, slice and renormalize the result yourself
— that is the third column, and it is a different quantity.

In [ ]:
windowed = MVR_CHMM(
    hidden_markov_model=hmm,
    constraints=[reach_mvr("B", time_range=[3, 6], name="reach_B_late")],
)

w_times, w_probs = sat_time_torch_mvr_chmm(windowed, observed, target="reach_B_late")

tail = probs.numpy()[3:]
sliced = tail / tail.sum()

print(pd.DataFrame({
    "t": w_times,
    "time_range=[3,6]": np.round(w_probs.numpy(), 4),
    "full, then sliced": np.round(sliced, 4),
}).to_string(index=False))

## Notes

- **First satisfaction, not per-time satisfaction.** Only the "not yet accepted"
  branch propagates, so the result is a stopping-time distribution over the
  target's window. The two coincide for a prefix-free target, which is what
  `mvr_sattime` produces.
- **The target is not required to hold at the end of its window**, unlike every
  constraint in ordinary constrained inference. Every *other* constraint is
  enforced as usual.
- A `time_range` on the target **initializes** the automaton at the window start
  rather than discarding early acceptances. Slice the result yourself if that is
  what you meant.
- `logsumexp(log_weights)` is the evidence
  $\log P(y,\ \text{others hold},\ \text{target satisfied in window})$, so the
  normalization discards genuinely useful information. Ask for it when you want
  the satisfaction probability rather than only its timing.
- `target` accepts an index into `model.constraints` (negatives allowed) or an
  MVR `name`. Names are set on the MVR instance, so a constraint built by an
  `@mvr_constraint_fn` factory is selectable by name only if the factory named
  what it built.
- Everything runs in log space. These weights are constraint-satisfaction
  probabilities, which decay exponentially with the horizon — at `T = 5000` they
  reach about `-3477`, where probability space would have flushed to zero by
  `T ~ 150`.
- The forward pass runs over the full augmented space; the backward pass drops
  the target's mediation axis, since past the satisfaction time it is no longer
  tracked.